Download LLM & prepare prompt for RAG

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# ---------------- Configuration ----------------

model_id = "Qwen/Qwen2-0.5B-Instruct"

print(f"Loading model: {model_id}...")

# ---------------- Load Model & Tokenizer ----------------

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",      # Use float16 if GPU available
    device_map="auto"        # Automatically use GPU if present, else CPU
)

# ---------------- Create Hugging Face Pipeline ----------------

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    temperature=0.1,
    do_sample=True
)

# ---------------- LangChain LLM Wrapper ----------------

raw_llm = HuggingFacePipeline(pipeline=pipe)

# ---------------- Prompt Formatter ----------------

def format_for_qwen(input_dict):
    messages = [
        {
            "role": "system",
            "content": """
You are a helpful AI assistant that answers questions strictly based on the provided context.
Do not use any external knowledge or make up information.
If the answer is not in the context, respond exactly with:
"I don't know based on the provided context."
Always start your response with "Answer:" followed by the answer or the "I don't know" statement.
"""
        },
        {
            "role": "user",
            "content": f"""
Context:
{input_dict['context']}

Question:
{input_dict['question']}
"""
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return formatted_prompt

# ---------------- Generation Function ----------------

def generate_with_qwen(formatted_prompt):
    response = raw_llm.invoke(formatted_prompt)

    generated = response.split(formatted_prompt)[-1].strip()

    if generated.startswith("Answer:"):
        generated = generated.split("Answer:", 1)[-1].strip()

    return generated

# ---------------- Create LLM Chain ----------------

llm_with_format = (
    RunnableLambda(format_for_qwen)
    | RunnableLambda(generate_with_qwen)
    | StrOutputParser()
)

print("LLM setup complete!")

Loading model: Qwen/Qwen2-0.5B-Instruct...


d:\AI-Course(DSTP3.0-BATCH-03)\Week11\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Munesh Kumar\.cache\huggingface\hub\models--Qwen--Qwen2-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 826.57it/s] 
[transformers] Passing `

LLM setup complete!


Building a RAG Pipeline

In [3]:
from langchain_core.runnables import RunnablePassthrough
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# --- 1. Define the Retriever ---

retriever = db.as_retriever(search_kwargs={"k": 2})

NameError: name 'db' is not defined

In [ ]:
# --- 2. Create the RAG Chain ---
# Retrieval -> Format context & question -> LLM -> Output

rag_chain = (
    {
        "context": retriever | (lambda docs: "\n\n".join(doc.page_content for doc in docs)),
        "question": RunnablePassthrough()
    }
    | llm_with_format
)

In [ ]:
# --- 3. Query Example ---

question = "What does RAG stand for?"

print(f"Question:\n{question}\n")

# Invoke the chain
response = rag_chain.invoke(question)

print("RAG Answer:")
print(response)

print("\n--- Context Used ---")

# Corrected: Use .invoke() on the retriever to get documents
context_docs = retriever.invoke(question)

for doc in context_docs:
    print(doc.page_content)